<a href="https://colab.research.google.com/github/nabinjoshi54/lis5693/blob/main/final-project/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install and Import Libraries

First, I install all the required packages. This project uses:
- `pandas` and `numpy` for data handling
- `nltk` for text preprocessing and sentiment analysis (VADER)
- `gensim` for LDA topic modeling
- `pyLDAvis` for interactive topic visualization
- `altair` and `vegafusion` for descriptive visualizations
- `scikit-learn` for machine learning models
- `seaborn` and `matplotlib` for confusion matrix plots
- `wordcloud` for word cloud visualization

In [ ]:
!pip install gensim pyLDAvis nltk altair vegafusion wordcloud scikit-learn vl-convert-python -q

import warnings
warnings.filterwarnings('ignore')

# For Jupyter internal warnings
import logging
logging.getLogger('jupyter_client').setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, CoherenceModel

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

import altair as alt
alt.data_transformers.enable('vegafusion')

from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('All libraries imported successfully!')

---
## Step 2: Load Dataset

The dataset was collected from [Lens.org](https://www.lens.org), an open-access scholarly literature platform. I am inspired by Lab-5 work so that I choose to enxtend lab5. I searched for three distinct research topics and exported the results as separate CSV files:

| Search Query | Category Label | Documents |
|---|---|---|
| `battery AND energy storage AND lithium` | `battery` | 608 |
| `space AND satellite AND spacecraft` | `space` | 608 |
| `text mining AND materials science AND battery` | `text_mining_materials` | 184 |
| **Total** | | **1,400** |

The three files were combined and saved as `final_dataset.csv` with a `category` label column added. The `Abstract` column is our primary text column for all analysis.

**Note on class imbalance:** The `text_mining_materials` category has fewer documents (184) compared to the other two (608 each). This is a natural reflection of how niche this research area is. We will handle this in the machine learning step using `class_weight='balanced'`.

In [ ]:
# Load the combined dataset from GitHub
import pandas as pd

url = "https://raw.githubusercontent.com/nabinjoshi54/lis5693/refs/heads/main/final-project/final_dataset.csv"
df = pd.read_csv(url)

print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nCategory distribution:')
print(df['category'].value_counts())

print('\nMissing values in Abstract column:', df['Abstract'].isna().sum())

df.head()

In [ ]:
# Keep only rows with non-empty abstracts and reset index
df = df[df['Abstract'].notna() & (df['Abstract'].str.strip() != '')].reset_index(drop=True)
print('Total usable documents:', len(df))
print('Category counts after cleaning:')
print(df['category'].value_counts())

## Step2: Reflection
In step 2, I checked how big the dataset is,are the categories correct, and does every paper have an abstract? Each paper (of total 1400 articles) has 33 pieces of information, which are title, author, journal, year, abstract, category... All 1400 articles have abstract in it so we do not have to remove any article that does not have abstract. Moreover, we printed first five rows of dataset to see what it looks like. This is basically data analysis before jumping straight to analysis.    

---
## Step 3: Text Preprocessing

Before any analysis, I preprocess the raw abstract text. This is a critical step because raw text contains noise — punctuation, numbers, and common words like "the" or "is" — that do not contribute meaningful information to topic modeling or classification.

The preprocessing pipeline includes:
1. **Lowercasing** — converts all text to lowercase so "Battery" and "battery" are treated as the same word
2. **Punctuation and number removal** — removes special characters and digits
3. **Tokenization** — splits text into individual words (tokens)
4. **Stopword removal** — removes common English words (e.g., "the", "and", "is") plus custom academic stopwords
5. **Lemmatization** — reduces words to their base form (e.g., "batteries" → "battery", "studying" → "study")

I also add custom stopwords that are very common in academic writing but do not help distinguish topics, such as "study", "result", "method", and "paper".

In [ ]:
# Set up stopwords and lemmatizer
stop_words = set(stopwords.words('english'))

# Custom stopwords: words common in all academic abstracts that add no topic signal
custom_stopwords = {
    'et', 'al', 'using', 'used', 'use', 'based', 'study', 'results',
    'method', 'methods', 'analysis', 'research', 'paper', 'article',
    'also', 'within', 'however', 'can', 'may', 'one', 'two', 'new',
    'show', 'shown', 'different', 'proposed', 'present', 'this',
    'however', 'although', 'furthermore', 'therefore', 'thus'
}
stop_words = stop_words.union(custom_stopwords)

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    """Full preprocessing pipeline: lowercase, remove noise, tokenize, remove stopwords, lemmatize."""
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove numbers
    text = re.sub(r'\d+', ' ', text)
    # Step 3: Remove punctuation
    text = re.sub(rf'[{re.escape(string.punctuation)}]', ' ', text)
    # Step 4: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 5: Tokenize
    tokens = text.split()
    # Step 6: Remove stopwords and short tokens
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2 and w.isalpha()]
    # Step 7: Lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens

print('Preprocessing pipeline defined.')
print('Sample test:')
sample = df['Abstract'].iloc[0]
print('\nOriginal (first 200 chars):')
print(sample[:200])
print('\nAfter preprocessing (first 20 tokens):')
print(preprocess(sample)[:20])

In [ ]:
# Apply preprocessing to all abstracts
# This creates two columns:
# 'tokens'     -> list of cleaned tokens (for LDA)
# 'clean_text' -> cleaned text as a single string (for TF-IDF and VADER)

print('Preprocessing all abstracts.')
df['tokens'] = df['Abstract'].apply(preprocess)
df['clean_text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))

# Remove any documents that ended up empty after preprocessing
df = df[df['tokens'].apply(len) > 0].reset_index(drop=True)

print(f'Documents after preprocessing: {len(df)}')
print('\nSample cleaned tokens (doc 0):')
print(df['tokens'].iloc[0][:15])

## Step-3: Reflection
This step 3 is all about thext preprocessing. As we all know that regular scientific literature contains nouns, pronouns, connector words etc which help make understand english as it is. However, these stopwords does not contribute meaningfully while performing topic modeling to extract some meaningful information. After removing numbers, punctuations, making every word lowecase etc, we make the dataset, which we suppose contains only relevant information, ready for further analysis. As you can see in the results, shown above, the unprocessed data has words like "the, is, with etc". After preprocessing we got only concrete technical words such as "spacecraft, mission, ionospheric etc". This step-3 preserves the quality data for further analysis.

---
## Step 4: Exploratory Data Analysis (EDA)

Before running any models, I perform exploratory data analysis to understand the structure and content of the dataset. I use **Altair** for all visualizations, as required by the course instructions.

The EDA covers:
- Distribution of documents by category
- Distribution of abstract lengths
- Top 30 most frequent words across the entire corpus
- Word clouds per category to visually compare vocabulary

In [ ]:
# --- Chart 1: Category Distribution ---
# This shows how many documents we have per category

cat_counts = df['category'].value_counts().reset_index()
cat_counts.columns = ['category', 'count']

chart_cat = alt.Chart(cat_counts).mark_bar().encode(
    x=alt.X('category:N', title='Research Category', sort='-y'),
    y=alt.Y('count:Q', title='Number of Documents'),
    color=alt.Color('category:N', legend=alt.Legend(title='Category')),
    tooltip=['category', 'count']
).properties(
    title='Document Count by Research Category',
    width=400,
    height=300
)

chart_cat

The bar chart above confirms our dataset has two balanced categories because both battery and space have each with 608 documents and one smaller category : text_mining_materials with 184 documents. This class imbalance will be accounted for during machine learning.

In [ ]:
# --- Chart 2: Abstract Length Distribution ---
# Abstract length (in words) tells us how rich our text data is

df['abstract_length'] = df['Abstract'].apply(lambda x: len(x.split()))

chart_len = alt.Chart(df).mark_bar(opacity=0.7).encode(
    x=alt.X('abstract_length:Q', bin=alt.Bin(maxbins=40), title='Abstract Length (words)'),
    y=alt.Y('count():Q', title='Number of Documents'),
    color=alt.Color('category:N', legend=alt.Legend(title='Category')),
    tooltip=['category', 'count()']
).properties(
    title='Distribution of Abstract Length by Category',
    width=500,
    height=300
)

chart_len

This stacked histogram, shown above, showed thre relationship between length of each abstract in words and how many documents fall in this category. The results showed that most results are between 100 to 220. The category 'battery' with blue color tend to have longer abstracts, followed by 'space' category with orange color.

In [ ]:
# --- Chart 3: Top 30 Most Frequent Words ---
# After preprocessing, which words appear most often across the entire corpus?

from collections import Counter

all_tokens = [token for tokens in df['tokens'] for token in tokens]
word_freq = Counter(all_tokens).most_common(30)
word_df = pd.DataFrame(word_freq, columns=['word', 'frequency'])

chart_words = alt.Chart(word_df).mark_bar().encode(
    x=alt.X('frequency:Q', title='Frequency'),
    y=alt.Y('word:N', sort='-x', title='Word'),
    color=alt.Color('frequency:Q', scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['word', 'frequency']
).properties(
    title='Top 30 Most Frequent Words (after preprocessing)',
    width=400,
    height=500
)

chart_words

The horizontal bar chart, shown above, showed the most frequently used words across 1400 abstracts. The word 'battery' is repeated most about 3000 times. 'energy', 'satellite', 'lithium' used most.

In [ ]:
# --- Chart 4: Word Clouds per Category ---
# Word clouds give an immediate visual sense of the dominant vocabulary
# in each research category

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
categories = df['category'].unique()
colors = ['Blues', 'Oranges', 'Greens']

for ax, cat, cmap in zip(axes, categories, colors):
    cat_text = ' '.join(df[df['category'] == cat]['clean_text'].tolist())
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap=cmap,
        max_words=80
    ).generate(cat_text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Category: {cat}', fontsize=13, fontweight='bold')

plt.suptitle('Word Clouds by Research Category', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

This word cloud gives the catching visual representation of how frequently the words appear in each category. We have three category with different colors. THe bigger the word appears, the more frequently it occurs in that category's abstracts.
The 'space' category is dominated by words like spacecraft, satellite, orbit, mission, operation etc. These terms are all related to articles realted to space.
The 'text mining materials' category is dominated by words like material, model, machine learning, application, information etc. This also proved that the article that discusses about text mining application in materials science have in fact discussed or mentioned these topics.
Finally, in 'battery' category, words like lithium ion, battery, electrolyte, storage system, renewable energy, energy density apperas frequently. This also proved that only technical terms related to core concepts related to battery research is present here.
This vocabulary separation gives us confidence that our machine learning models will perform well.


---
## Task 1 — Text Analysis Method 1: Topic Modeling (LDA)

### What is LDA?
Latent Dirichlet Allocation (LDA) is an unsupervised method that discovers hidden thematic structure in a collection of documents. Each document is modeled as a mixture of topics, and each topic is characterized by a distribution of words. This is very similar to what Tshitoyan et al. (2019) did — they used Word2Vec to capture co-occurrence patterns in abstracts, and LDA does the same by finding which words tend to appear together.

### Why LDA over Network Analysis or other methods?
LDA is the most appropriate method for this dataset because:
1. Our documents are scientific abstracts — structured, content-rich text ideal for topic discovery
2. We want to understand *what themes* exist across 3 research domains, not just *which words* are connected
3. LDA directly mirrors the paper's goal: extracting latent knowledge from scientific literature

I will train models with 5, 10, and 15 topics and use the **coherence score** to select the best model.

In [ ]:
# --- Build Gensim Dictionary and Corpus ---
# The dictionary maps each unique token to a numeric ID
# The corpus converts each document into a bag-of-words representation

cleaned_docs = df['tokens'].tolist()

# Build dictionary
dictionary = corpora.Dictionary(cleaned_docs)

# Filter extremes: remove words that appear in fewer than 5 docs or more than 50% of docs
# This removes very rare and very common words that don't help distinguish topics
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Convert to bag-of-words corpus
corpus = [dictionary.doc2bow(doc) for doc in cleaned_docs]

print('Number of unique tokens in dictionary:', len(dictionary))
print('Number of documents in corpus:', len(corpus))
print('Sample BOW for document 0 (first 5 entries):', corpus[0][:5])

Before training the LDA model, the cleaned tokesn are converted into two structures required by Gensim. The dictionary assigns a unique numeric ID to each of the 2,951 meaningful words in the corpus. Words that are too rare (appearing in fewer than 5 documents) or too common (appearing in more than 50% of documents) are filtered out, as they do not contribute useful topic signal. Each document is then converted into a bag-of-words representation — a list of (word_id, frequency) pairs — which is the input format LDA requires.

In [ ]:
# --- Train LDA Models with 5, 10, and 15 Topics ---
# We train three models and compare them using coherence scores

print('Training LDA models... this may take a few minutes.')

lda_5 = LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=5,
    random_state=42, passes=10, alpha='auto', per_word_topics=True
)
print('LDA 5-topic model trained.')

lda_10 = LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=10,
    random_state=42, passes=10, alpha='auto', per_word_topics=True
)
print('LDA 10-topic model trained.')

lda_15 = LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=15,
    random_state=42, passes=10, alpha='auto', per_word_topics=True
)
print('LDA 15-topic model trained.')

Three LDA models are trained with 5, 10, and 15 topics respectively to determine the optimal number of topics for this corpus. Each model uses 10 passes through the entire corpus to ensure stable convergence, and alpha='auto' allows the model to automatically learn the topic distribution per document. The same random_state=42 is used across all models to ensure reproducibility of results.

In [ ]:
# --- Print Top Words for Each Model ---

def print_topics(model, num_words=8):
    for idx, topic in model.print_topics(num_words=num_words):
        print(f'  Topic {idx}: {topic}')

print('===== 5 TOPICS =====')
print_topics(lda_5)

print('\n===== 10 TOPICS =====')
print_topics(lda_10)

print('\n===== 15 TOPICS =====')
print_topics(lda_15)

The top 8 words for each topic across all three models are printed for inspection. These keywords allow us to manually interpret what real-world theme each topic represents. For example, in the 10-topic model, Topic 3 (battery, energy, storage, lithium, system, ion, power, technology) clearly represents battery energy storage systems, while Topic 2 (satellite, space, spacecraft, mission, orbit, earth) clearly represents space mission research. Topic 8 (data, material, model, synthesis, text, scientific, information) represents the text mining applied to materials science — the hybrid third category. This manual interpretation is a key strength of LDA over black-box methods.

In [ ]:
# --- Coherence Score Comparison ---
# Coherence score (c_v) measures how semantically similar the top words
# within each topic are. Higher coherence = more interpretable topics.

print('Calculating coherence scores...')

cv_5  = CoherenceModel(model=lda_5,  texts=cleaned_docs, dictionary=dictionary, coherence='c_v').get_coherence()
cv_10 = CoherenceModel(model=lda_10, texts=cleaned_docs, dictionary=dictionary, coherence='c_v').get_coherence()
cv_15 = CoherenceModel(model=lda_15, texts=cleaned_docs, dictionary=dictionary, coherence='c_v').get_coherence()

print(f'Coherence Score  5 topics: {cv_5:.4f}')
print(f'Coherence Score 10 topics: {cv_10:.4f}')
print(f'Coherence Score 15 topics: {cv_15:.4f}')

The coherence score (c_v) measures the semantic similarity of the top words within each topic — higher scores indicate more interpretable and meaningful topics. The 10-topic model achieves the highest coherence score of 0.4883, outperforming both the 5-topic model (0.4445) and the 15-topic model (0.4725). This result makes intuitive sense because 5 topics are too few to capture the distinct subthemes within battery and space research, while 15 topics introduces redundancy as some topics begin to overlap. The 10-topic model is therefore selected as the best model for further analysis and visualization. This finding aligns with the principle that the optimal number of topics balances granularity with interpretability.

In [ ]:
# --- Altair Bar Chart: Coherence Scores ---

coherence_df = pd.DataFrame({
    'Model': ['5 Topics', '10 Topics', '15 Topics'],
    'Coherence Score': [cv_5, cv_10, cv_15]
})

chart_coherence = alt.Chart(coherence_df).mark_bar().encode(
    x=alt.X('Model:N', title='LDA Model', sort=['5 Topics', '10 Topics', '15 Topics']),
    y=alt.Y('Coherence Score:Q', title='Coherence Score (c_v)', scale=alt.Scale(zero=False)),
    color=alt.Color('Model:N', legend=None),
    tooltip=['Model', 'Coherence Score']
).properties(
    title='LDA Coherence Score Comparison',
    width=350,
    height=300
)

# Add text labels on bars
text = chart_coherence.mark_text(dy=-8).encode(
    text=alt.Text('Coherence Score:Q', format='.4f')
)

(chart_coherence + text)

This bar chart compares the coherence scores of three LDA models and plotted here.

### Model Selection

Based on the coherence scores and visual inspection of the topic keywords, I select the model that provides the most interpretable and distinct topics. A higher coherence score indicates more semantically coherent topics, but we also consider interpretability — topics should make intuitive sense given our three research domains (battery, space, text mining in materials science).

The selected model is used for the pyLDAvis visualization below.

In [ ]:
# --- Select Best Model (based on coherence + interpretability) ---
# Changing 'best_lda' to lda_10 based on my coherence results

best_lda = lda_10  # Update this after reviewing your coherence scores

print('Selected model: 10 topics')
print('Top 10 words per topic in selected model:\n')
for idx, topic in best_lda.print_topics(num_words=10):
    print(f'Topic {idx}: {topic}')
    print()

After finding that lda_10 is the best model, we choose it and displays 10 topic models with top 10 words each.

In [ ]:
# --- pyLDAvis Interactive Visualization ---
# Each circle represents a topic. Circle size = topic prevalence.
# Distance between circles = topic distinctness.
# The bar chart on the right shows top words for the selected topic.

vis = gensimvis.prepare(best_lda, corpus, dictionary)
vis

Circle size on left represents how prevalent the topic is across all 1400 documents. Bigger circle means more documents belong to that topic. The distance between circles means that the topics are different from each other and share very few words. If we put mouse cursor on each circle, it shows the the top 30 terms for that topic on right bar chart.

### Topic Interpretation

Based on the top keywords from the selected LDA model, I assign the following topic labels:

- **Topic 0:** *( Battery Energy Storage & Electric Vehicles)*
- **Topic 1:** *( Space Missions & Orbital Operations)*
- **Topic 2:** *( Lithium-Ion Battery Chemistry & Materials)*
- **Topic 3:** *( Space Environment & Radiation Effects)*
- **Topic 4:** *( Materials Science Text Mining & Informatics)*
- **Topic 5:** *(Battery Management Systems & Fault Detection)*
- **Topic 6:** *(Satellite Engineering & Systems Control)*
- **Topic 7:** *(Machine Learning for Materials Discovery)*
- **Topic 8:** *(Satellite Communication Networks)*
- **Topic 9:** *(Space Science Instrumentation & Sensing)*

The pyLDAvis visualization above confirms that the topics are well-separated (non-overlapping circles), indicating good topic distinctness.

---
## Task 1 — Text Analysis Method 2: Sentiment Analysis (VADER)

### What is VADER?
VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analysis tool from NLTK. It assigns four scores to any text: positive, negative, neutral, and a compound score (ranging from -1 to +1). A compound score above 0.05 is generally considered positive, below -0.05 is negative, and between -0.05 and 0.05 is neutral.

### Why Sentiment Analysis?
Scientific abstracts often make claims like *"we achieve superior performance"* or *"this novel approach significantly improves..."* — language that carries positive sentiment. By comparing sentiment scores across our three research categories, we can discover:
- Do battery papers make more confident claims than space papers?
- Does the text mining + materials science domain use more cautious language?
- How has sentiment in these fields evolved over time?

This is a unique angle that complements the structural analysis done by LDA.

In [ ]:
# --- Generate VADER Sentiment Scores ---
# We use the raw Abstract text (not preprocessed) for sentiment analysis
# because VADER relies on punctuation and capitalization for scoring

print('Running VADER sentiment analysis on all abstracts...')
sid = SentimentIntensityAnalyzer()

scores_list = []
for idx, row in df.iterrows():
    scores = sid.polarity_scores(row['Abstract'])
    scores['Lens ID'] = row['Lens ID']
    scores['category'] = row['category']
    scores['Publication Year'] = row.get('Publication Year', None)
    scores_list.append(scores)

sentiment_df = pd.DataFrame(scores_list)

# Add sentiment label based on compound score
def label_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

sentiment_df['sentiment_label'] = sentiment_df['compound'].apply(label_sentiment)

print('Sentiment analysis complete!')
print('\nSummary statistics for compound score:')
print(sentiment_df['compound'].describe())
print('\nOverall sentiment distribution:')
print(sentiment_df['sentiment_label'].value_counts())

VADER sentiment analysis was applied to all 1,400 raw abstracts. The results reveal a strongly positive sentiment profile across the entire corpus. The mean compound score is 0.574 and the median is 0.827, indicating that the majority of scientific abstracts carry strong positive sentiment. Of the 1,400 documents, 1,165 (83.2%) are classified as Positive, 176 (12.6%) as Negative, and only 59 (4.2%) as Neutral. This overwhelming positivity is consistent with academic writing conventions — researchers frame their abstracts to emphasize novelty, achievement, and improvement in order to attract readers and convince reviewers of their work's significance. This finding directly supports the observation made by Tshitoyan et al. (2019) that scientific abstracts tend to emphasize positive relationships, which they noted actually aided their Word2Vec predictions. The wide standard deviation (0.518) indicates substantial variation — some papers are extremely positive while others carry genuinely negative sentiment, likely those reporting on system failures, safety risks, or field-wide challenges.

In [ ]:
# --- Chart: Mean Compound Sentiment Score by Category ---

mean_sentiment = sentiment_df.groupby('category')['compound'].mean().reset_index()
mean_sentiment.columns = ['category', 'mean_compound']

chart_sent_cat = alt.Chart(mean_sentiment).mark_bar().encode(
    x=alt.X('category:N', title='Research Category', sort='-y'),
    y=alt.Y('mean_compound:Q', title='Mean Compound Sentiment Score'),
    color=alt.Color('category:N', legend=alt.Legend(title='Category')),
    tooltip=['category', 'mean_compound']
).properties(
    title='Mean Sentiment Score by Research Category',
    width=400,
    height=300
)

text_sent = chart_sent_cat.mark_text(dy=-8).encode(
    text=alt.Text('mean_compound:Q', format='.3f')
)

(chart_sent_cat + text_sent)

Battery and text mining materials papers score significantly higher in sentiment (0.751 and 0.724 respectively) compared to space papers (0.351). Battery researchers frequently use confident, achievement-oriented language to report performance improvements, while space engineering papers adopt a more cautious, risk-aware tone — regularly discussing system anomalies, radiation hazards, and orbital debris. This reveals meaningful rhetorical differences between scientific communities beyond just topical differences, and is consistent with Tshitoyan et al.'s observation that scientific abstracts tend to emphasize positive relationships.

In [ ]:
# --- Chart: Box Plot of Compound Scores by Category ---
# Box plots show the full distribution, not just the mean

chart_box = alt.Chart(sentiment_df).mark_boxplot(extent='min-max', size=50).encode(
    x=alt.X('category:N', title='Research Category'),
    y=alt.Y('compound:Q', title='Compound Sentiment Score'),
    color=alt.Color('category:N', legend=alt.Legend(title='Category')),
    tooltip=['category', 'compound']
).properties(
    title='Distribution of Sentiment Scores by Category',
    width=400,
    height=350
)

chart_box

The box plot reveals not just the average sentiment but the full distribution within each category. Battery and text mining materials both show narrow, high-positioned boxes, indicating that papers in these fields consistently use positive language with little variation. Space papers show a dramatically wider box spanning nearly the full sentiment range, reflecting the coexistence of highly positive mission success papers alongside neutral or negative papers reporting technical failures, radiation risks, and system anomalies. This variability in space research sentiment is itself a meaningful finding — space is a more rhetorically diverse field than battery research.

In [ ]:
# --- Chart: Sentiment Label Distribution by Category ---
# How many abstracts are positive, neutral, or negative per category?

sent_label_counts = sentiment_df.groupby(['category', 'sentiment_label']).size().reset_index(name='count')

chart_sent_labels = alt.Chart(sent_label_counts).mark_bar().encode(
    x=alt.X('category:N', title='Research Category'),
    y=alt.Y('count:Q', title='Number of Documents'),
    color=alt.Color('sentiment_label:N',
                    scale=alt.Scale(domain=['Positive', 'Neutral', 'Negative'],
                                    range=['#2ecc71', '#95a5a6', '#e74c3c']),
                    legend=alt.Legend(title='Sentiment')),
    tooltip=['category', 'sentiment_label', 'count']
).properties(
    title='Sentiment Label Distribution by Category',
    width=400,
    height=300
)

chart_sent_labels

The stacked bar chart confirms the sentiment differences across categories at the document level. Battery and text mining materials papers are overwhelmingly positive (~92% and ~95% respectively), with very few negative documents. Space papers tell a different story — while still majority positive (71%), they contain a substantially larger proportion of negative papers ~23%, visible as the prominent red section. This confirms that space research encompasses a broader emotional range, with a significant share of papers discussing failures, hazards, and technical challenges in genuinely negative terms. This chart provides count-level evidence supporting the mean score differences observed in the previous chart.


In [ ]:
# --- Chart: Mean Sentiment Over Publication Years ---
# Has the tone of scientific abstracts changed over time?

sentiment_df['Publication Year'] = pd.to_numeric(sentiment_df['Publication Year'], errors='coerce')
yearly = sentiment_df.dropna(subset=['Publication Year'])
yearly = yearly[yearly['Publication Year'] >= 2000]
yearly_mean = yearly.groupby(['Publication Year', 'category'])['compound'].mean().reset_index()

chart_year = alt.Chart(yearly_mean).mark_line(point=True).encode(
    x=alt.X('Publication Year:O', title='Publication Year'),
    y=alt.Y('compound:Q', title='Mean Compound Sentiment Score'),
    color=alt.Color('category:N', legend=alt.Legend(title='Category')),
    tooltip=['Publication Year', 'category', 'compound']
).properties(
    title='Mean Sentiment Score Over Time by Category',
    width=600,
    height=300
)

chart_year

The line chart tracks mean sentiment over time from 2000 to 2026. Battery papers (blue) remain consistently and stably positive throughout the entire period, confirming that positive, achievement-oriented language is a deeply ingrained norm in battery research regardless of the year. Text mining and materials papers (pink) follow a similar high-positive trend but with more year-to-year volatility, likely due to fewer documents per year making individual papers more influential on the annual average. Space papers (orange) show the most erratic pattern — fluctuating dramatically between near-negative and highly positive scores across years — reflecting the inherent diversity of space research topics published in any given year. Notably, all three categories show relatively higher sentiment scores in more recent years (2022–2025), which may reflect a general trend toward more optimistic scientific writing as these fields have matured and achieved more consistent results.

### Sentiment Analysis Interpretation

The sentiment analysis reveals interesting patterns across the three research categories:

- **Overall tone:** The majority of scientific abstracts are positive in sentiment, which is expected — researchers tend to emphasize the novelty and success of their work.
- **Category differences:** *(Fill in after running — e.g., battery papers tend to be more positive than space papers, possibly because battery research more often reports performance improvements)*
- **Over time:** *(Fill in after running — e.g., sentiment has become more positive over the years as the field has matured)*

This finding is consistent with the Tshitoyan et al. paper's observation that scientific abstracts emphasize positive relationships, which actually aided their Word2Vec predictions.

---
## Task 2: Machine Learning Classification

### Goal
Build a supervised machine learning model that can predict the **research category** of a scientific abstract (battery, space, or text_mining_materials) based on its text content.

### Feature Engineering: TF-IDF
I use **TF-IDF (Term Frequency-Inverse Document Frequency)** to convert the cleaned text into numerical feature vectors. TF-IDF assigns higher weights to words that are frequent in a document but rare across the corpus — this is perfect for distinguishing domain-specific vocabulary like *lithium* (common in battery papers but rare in space papers).

### Classifiers Compared
I train and compare three classifiers:
1. **Logistic Regression** — a simple linear model, baseline for text classification
2. **Random Forest** — an ensemble of decision trees, captures non-linear patterns
3. **SVM (LinearSVC)** — Support Vector Machine, known to work very well with TF-IDF features

All models use `class_weight='balanced'` to handle the class imbalance in `text_mining_materials`.

In [ ]:
# --- Feature Engineering: TF-IDF Vectorization ---

# Using cleaned text (preprocessed tokens joined back into string)
X_text = df['clean_text'].values
y = df['category'].values

# TF-IDF: max 5000 features, unigrams and bigrams (1,2)
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # include both single words and two-word phrases
    sublinear_tf=True     # apply log normalization to term frequency
)

X = tfidf.fit_transform(X_text)

print('TF-IDF matrix shape:', X.shape)
print('Number of features (unique terms):', X.shape[1])
print('Number of documents:', X.shape[0])

# Sample top TF-IDF terms
feature_names = tfidf.get_feature_names_out()
print('\nSample TF-IDF features:', feature_names[:20].tolist())

In [ ]:
# --- Train/Test Split ---
# 80% training, 20% testing
# stratify=y ensures each category is proportionally represented in both splits

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]} documents')
print(f'Test set size:     {X_test.shape[0]} documents')
print('\nTraining set category distribution:')
from collections import Counter
print(Counter(y_train))
print('\nTest set category distribution:')
print(Counter(y_test))

In [ ]:
# --- Model 1: Logistic Regression ---
# Logistic Regression is a strong baseline for text classification.
# It learns a linear decision boundary between categories in TF-IDF space.
# class_weight='balanced' adjusts for the class imbalance in text_mining_materials.

print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# 5-fold cross-validation for more reliable performance estimate
lr_cv = cross_val_score(lr, X, y, cv=5, scoring='f1_macro')

print('\n── Logistic Regression Results ──')
print(classification_report(y_test, lr_pred, target_names=['battery', 'space', 'text_mining_materials']))
print(f'5-fold CV F1 macro: {lr_cv.mean():.3f} ± {lr_cv.std():.3f}')

In [ ]:
# --- Confusion Matrix: Logistic Regression ---

fig, ax = plt.subplots(figsize=(6, 5))
cm_lr = confusion_matrix(y_test, lr_pred, labels=['battery', 'space', 'text_mining_materials'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                               display_labels=['battery', 'space', 'text_mining\nmaterials'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Model 2: Random Forest ---
# Random Forest builds many decision trees and combines their votes.
# It can capture non-linear patterns and interactions between terms.
# n_estimators=200 means 200 trees are built and averaged.

print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_cv = cross_val_score(rf, X, y, cv=5, scoring='f1_macro')

print('\n── Random Forest Results ──')
print(classification_report(y_test, rf_pred, target_names=['battery', 'space', 'text_mining_materials']))
print(f'5-fold CV F1 macro: {rf_cv.mean():.3f} ± {rf_cv.std():.3f}')

In [ ]:
# --- Confusion Matrix: Random Forest ---

fig, ax = plt.subplots(figsize=(6, 5))
cm_rf = confusion_matrix(y_test, rf_pred, labels=['battery', 'space', 'text_mining_materials'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['battery', 'space', 'text_mining\nmaterials'])
disp.plot(ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Model 3: SVM (LinearSVC) ---
# SVM is widely regarded as one of the best classifiers for text.
# It finds the optimal hyperplane that maximally separates categories
# in the high-dimensional TF-IDF space.

print('Training SVM (LinearSVC)...')
svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

svm_cv = cross_val_score(svm, X, y, cv=5, scoring='f1_macro')

print('\n── SVM (LinearSVC) Results ──')
print(classification_report(y_test, svm_pred, target_names=['battery', 'space', 'text_mining_materials']))
print(f'5-fold CV F1 macro: {svm_cv.mean():.3f} ± {svm_cv.std():.3f}')

In [ ]:
# --- Confusion Matrix: SVM ---

fig, ax = plt.subplots(figsize=(6, 5))
cm_svm = confusion_matrix(y_test, svm_pred, labels=['battery', 'space', 'text_mining_materials'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                               display_labels=['battery', 'space', 'text_mining\nmaterials'])
disp.plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Confusion Matrix — SVM (LinearSVC)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Model Comparison Summary ---

summary_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM (LinearSVC)'],
    'CV F1 Macro (mean)': [lr_cv.mean(), rf_cv.mean(), svm_cv.mean()],
    'CV F1 Macro (std)': [lr_cv.std(), rf_cv.std(), svm_cv.std()]
})
summary_df['CV F1 Macro (mean)'] = summary_df['CV F1 Macro (mean)'].round(3)
summary_df['CV F1 Macro (std)'] = summary_df['CV F1 Macro (std)'].round(3)
print(summary_df.to_string(index=False))

# Altair chart for comparison
chart_compare = alt.Chart(summary_df).mark_bar().encode(
    x=alt.X('Model:N', title='Classifier'),
    y=alt.Y('CV F1 Macro (mean):Q', title='F1 Macro Score (5-fold CV)',
            scale=alt.Scale(zero=False)),
    color=alt.Color('Model:N', legend=None),
    tooltip=['Model', 'CV F1 Macro (mean)', 'CV F1 Macro (std)']
).properties(
    title='Model Comparison: 5-Fold Cross-Validation F1 Macro Score',
    width=400,
    height=300
)

text_compare = chart_compare.mark_text(dy=-8).encode(
    text=alt.Text('CV F1 Macro (mean):Q', format='.3f')
)

(chart_compare + text_compare)

### Best Classifier Selection

Based on the 5-fold cross-validation F1 macro scores, **Logistic Regression** is selected as the best classifier. It achieves the highest F1 macro score with the lowest variance, meaning it is both the most accurate and the most stable across different subsets of the data.

This result is consistent with established text classification literature — Logistic Regression combined with TF-IDF features is a strong, well-understood baseline that often outperforms more complex models on well-separated text categories. The high performance (F1 > 0.97) reflects the fact that our three research domains use highly distinct vocabulary, making classification relatively straightforward.

### Key Observations
1. **Battery and space categories** are classified almost perfectly (F1 ~0.98–0.99) because their vocabulary is very domain-specific and non-overlapping.
2. **text_mining_materials** has slightly lower recall because it is a hybrid topic — it borrows vocabulary from both battery research and NLP/text mining, making it harder to distinguish.
3. **Class imbalance** was successfully handled using `class_weight='balanced'`, preventing the model from ignoring the smaller category.

---
## Conclusion and Discussion

This project applied a full text mining and machine learning pipeline to 1,400 scientific abstracts collected from Lens.org across three research domains.

### Key Findings

**Topic Modeling (LDA):**
- The LDA model successfully uncovered distinct themes within the corpus, including battery chemistry, spacecraft power systems, energy storage management, and NLP-based materials discovery
- Topics aligned closely with the three dataset categories, validating our dataset construction
- This mirrors the Tshitoyan et al. paper's finding that scientific abstracts contain rich latent structure that unsupervised methods can extract

**Sentiment Analysis (VADER):**
- The majority of scientific abstracts carry positive sentiment, consistent with the academic tendency to highlight novelty and achievement
- Sentiment scores differ across categories, suggesting different rhetorical styles across research domains
- This finding supports Tshitoyan et al.'s observation that scientific abstracts emphasize positive relationships

**Machine Learning:**
- Logistic Regression achieved the best performance (F1 macro ~0.979) and was selected as the best classifier
- All three classifiers performed well, confirming that TF-IDF captures the vocabulary differences between categories effectively
- Class imbalance in `text_mining_materials` was successfully mitigated using balanced class weights

### Limitations
1. The dataset is limited to three research domains — a broader dataset would produce more generalizable models
2. VADER was designed for social media text and may not fully capture the nuance of academic writing
3. TF-IDF does not capture word order or semantic similarity — future work could use Word2Vec or BERT embeddings as in the Tshitoyan et al. paper
4. The `text_mining_materials` class imbalance (184 vs 608 documents) may still affect generalizability

### Future Work
- Train Word2Vec embeddings on this corpus (as done in the paper) and use them as features for the ML models
- Expand the dataset with more categories (e.g., solar energy, robotics) for a richer classification task
- Apply BERT or other transformer-based models for more accurate sentiment analysis on scientific text